In [1]:
# 1. Import libraries and helper functions

import pandas as pd
from pathlib import Path
import os

def normalize_text(series):
    return series.astype(str).str.strip().str.title()

In [2]:
# 2. Path configuration

BASE_DIR = Path(os.environ.get("MASKING_DATA_PATH", "../data"))
RAW_DIR = BASE_DIR / "raw" / "us"
PROCESSED_DIR = BASE_DIR / "processed" / "us"

In [3]:
# 3. Load dataset

df = pd.read_excel(RAW_DIR / "target_raw.xlsx").dropna(how='all')

In [4]:
# 4. Data quality check

def data_quality_check(df, name="dataset", id_cols=None):
    """
    Runs a standard data quality check on any DataFrame.
    id_cols: str, list of str, or None. Pass the column(s) that should be
    unique keys - the function checks each one independently.
    """

    # 1. Header
    print(f"{'='*60}")
    print(f"DATA QUALITY CHECK: {name}")
    print(f"{'='*60}\n")

    # 2. Shape
    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns\n")

    # 3. Data types
    print("--- Data types ---")
    print(df.dtypes)
    print()

    # 4. Nulls
    print("--- Null values ---")
    nulls = df.isna().sum()
    print(nulls[nulls > 0] if nulls.sum() > 0 else "No nulls found")
    print()

    # 5. Fully duplicated rows (entire row identical)
    print("--- Duplicated rows (entire row) ---")
    print(df.duplicated().sum())
    print()

    # 6. Key column(s) duplicated on their own, regardless of the rest of the row
    if id_cols:
        if isinstance(id_cols, str):
            id_cols = [id_cols]
        for col in id_cols:
            if col not in df.columns:
                print(f"WARNING: id_col '{col}' not found in this dataset - skipping\n")
                continue
            print(f"--- Duplicated '{col}' (key column) ---")
            id_dupes = df[col].duplicated().sum()
            print(id_dupes)
            if id_dupes > 0:
                print(f"WARNING: {id_dupes} duplicate '{col}' found - "
                      f"inspect whether other columns diverge between them:")
                print(df[df[col].duplicated(keep=False)].sort_values(col).head(20))
            print()

    # 7. Categorical columns: unique values + top values (spot inconsistent text)
    print("--- Categorical columns: unique value counts ---")
    for col in df.select_dtypes(include='object').columns:
        print(f"\n{col}: {df[col].nunique()} unique values")
        print(df[col].value_counts().head(10))

    # 8. Numeric columns: descriptive stats (spot outliers, negatives, zeros)
    print("\n--- Numeric columns: descriptive stats ---")
    print(df.describe())

    print(f"\n{'='*60}\n")
    
data_quality_check(df, name="target_raw", id_cols=None)

DATA QUALITY CHECK: target_raw

Shape: 775 rows x 3 columns

--- Data types ---
DATE            datetime64[ns]
SALES_REP               object
TARGET_VALUE            object
dtype: object

--- Null values ---
No nulls found

--- Duplicated rows (entire row) ---
0

--- Categorical columns: unique value counts ---

SALES_REP: 25 unique values
SALES_REP
Abigail Shaffer         31
Holly Wood              31
Ryan Munoz              31
Patty Perez             31
Noah Rhodes             31
Monica Herrera          31
Michele Williams        31
Matthew Foster          31
Margaret Hawkins DDS    31
Lisa Jackson            31
Name: count, dtype: int64

TARGET_VALUE: 775 unique values
TARGET_VALUE
$41,612.18     1
$101,792.59    1
$265,697.39    1
$77,460.45     1
$229,486.51    1
$208,952.08    1
$127,927.44    1
$187,871.78    1
$259,590.55    1
$116,268.82    1
Name: count, dtype: int64

--- Numeric columns: descriptive stats ---
                                DATE
count                        

In [5]:
# 5. Rename columns

df.columns = df.columns.str.lower()

In [6]:
# 6. Add rep_id from the dimension table

reps_dim = pd.read_csv(PROCESSED_DIR / "dim_reps.csv")

df['sales_rep'] = normalize_text(df['sales_rep'])

df = df.merge(
    reps_dim[['rep_id', 'rep_name']],
    left_on='sales_rep',
    right_on='rep_name',
    how='left'
)

# Validate: every sales_rep name must have matched a rep_id
assert df['rep_id'].isna().sum() == 0, "Some sales_rep names did not match any rep_id!"

df = df.drop(columns=['rep_name'])

df = df[['date', 'sales_rep', 'rep_id', 'target_value']]

In [7]:
# 7. Clean column's values

# Remove '$' and thousand separators (commas), keep the decimal point

df['target_value'] = (
    df['target_value']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
    .astype(float)
)

In [8]:
# 8. Convert data types

df['rep_id'] = df['rep_id'].astype(str)
df['sales_rep'] = df['sales_rep'].astype('category')

In [9]:
# 9. Data quality check - duplicate records

# sales_rep + date should be unique - one target per rep, per month.
dupes = df.duplicated(subset=['sales_rep', 'date']).sum()
print(f"{dupes} duplicate sales_rep + date combinations found")


0 duplicate sales_rep + date combinations found


In [10]:
# 10. Final checks
assert dupes == 0, "Duplicate rep_id + date found in target table!"
assert df['rep_id'].isin(reps_dim['rep_id'].astype(str)).all(), "Orphan rep_id found!"

In [11]:
# 11. Export cleaned dataset for BI 

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(PROCESSED_DIR / "fact_target.csv", index=False)

print(f"Success! {len(df)} target records exported.")

Success! 775 target records exported.
